# JUDY: Local FM Propagation Modeling

**NWR Coverage Gap Research — Summer 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/nwr-gap-research/blob/main/judy_model_exploration.ipynb)

---

In Weeks 3–5 you used the RadioLand API to get field-strength predictions. That API runs a propagation model called **ITM (Irregular Terrain Model)** — a physics-based simulation that accounts for terrain, transmitter power, height, and frequency. Each call takes 15–30 seconds.

This notebook introduces a second model: **JUDY** — an XGBoost machine-learning model trained on ~24,000 real FM signal measurements. JUDY doesn't know anything about terrain; it learned propagation patterns directly from data. The tradeoff:

| | RadioLand API (ITM) | JUDY (XGBoost) |
|---|---|---|
| Speed | 15–30 sec / location | < 1 ms / location |
| Accuracy | High (physics-based + terrain) | Good (2.5–3.9 dBu MAE) |
| Frequency range | FM, AM, WX | FM only (88–108 MHz) |
| Requires server | Yes | No — runs locally |
| Runs offline | No | Yes |

The speed difference is what makes JUDY interesting for research: you can run **thousands of predictions per second** locally, enabling grid sweeps and sensitivity analyses that would take hours via the API.

**Learning goals:**
- Understand how an XGBoost regression model works at a high level
- Load and call JUDY for single and batch predictions
- Understand each input feature and its physical meaning
- Compare JUDY predictions against RadioLand API results to characterize model error
- Explore what JUDY can (and can't) tell us about NWR coverage

---
## Part 1: Setup

In [ ]:
# Install dependencies if needed (Colab only — local installs should already have these)
# !pip install xgboost joblib numpy pandas matplotlib folium requests

import os
import math
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# judy_predictor.py and the .joblib model file must be in the same directory as this notebook
from judy_predictor import JudyPredictor

judy = JudyPredictor()

if judy.ready:
    print("JUDY model loaded successfully.")
else:
    print("ERROR: JUDY model failed to load. Check that xgboost_field_strength_stratified.joblib is present.")

---
## Part 2: What JUDY Knows (and How It Learned It)

JUDY is an **XGBoost regression model** — a gradient-boosted ensemble of decision trees. It was trained on a dataset of ~24,000 FM station–receiver measurement pairs collected from real-world FCC propagation data via the RadioLand API.

For each training example, the model saw 16 input features describing the transmitter, receiver, and their geometry. The target variable was field strength in **dBu (dB-microvolts/meter)** — the same unit you've been using.

### The 16 Input Features

| Feature | Unit | Description |
|---|---|---|
| `frequency` | MHz | FM transmit frequency (training range: 88.1–108.1) |
| `erp` | watts | Effective Radiated Power |
| `haat` | meters | Height Above Average Terrain |
| `hagl` | meters | Height Above Ground Level at transmitter |
| `amsl` | meters | Transmitter altitude above mean sea level |
| `lat`, `lon` | degrees | Transmitter coordinates |
| `receiver_lat`, `receiver_lon` | degrees | Receiver coordinates |
| `distance_miles` | miles | TX–RX distance |
| `bearing` | degrees | Bearing *from receiver to transmitter* (0–360) |
| `antenna_type` | — | `"DRL"` = directional; anything else = omni |
| `aant_rotation_deg` | degrees | Antenna pattern rotation |
| `class_flag` | — | FCC class: `"A"`, `"B"`, `"B1"`, `"C"`, `"C0"`–`"C3"`, `"D"` |
| `uneven_polarization` | 0–1 | Polarization asymmetry |

### Preprocessing Pipeline

Before feeding features to the model, `judy_predictor.py` applies the same transforms used during training:
1. **Log transform** ERP: `log10(erp)` — compresses the wide dynamic range (0.1 W to 100 kW)
2. **Log transform** distance: `log(miles + 1)` — compresses the distance range
3. **Negate longitude**: forces western-hemisphere convention (negative)
4. **Encode antenna type**: `DRL → 1`, everything else → `0`
5. **Encode class flag**: `A → 10`, `B/B1 → 7.5`, `C/C0–C3 → 5`, `D → 2.5`
6. **Min-max normalize** all features to `[0, 1]` using the ranges from the training dataset

> **NWR note:** JUDY was trained on FM stations at 88.1–108.1 MHz. NWR operates at 162.4–162.55 MHz — well outside that range. The model clips NWR frequencies to the training maximum (108.1 MHz) after normalization. You'll quantify what this means for accuracy in Part 6.

---
## Part 3: Your First JUDY Prediction

Let's predict the field strength of a well-known FM station: **WXPN 88.5 Philadelphia**, at a receiver in Tabernacle, NJ — the same test location from Week 3.

In [ ]:
# WXPN 88.5, Philadelphia — transmitter parameters from FCC data
WXPN = dict(
    frequency          = 88.5,
    erp                = 18000,      # watts ERP
    haat               = 174,        # meters HAAT
    hagl               = 124,        # meters HAGL
    amsl               = 202,        # meters AMSL
    lat                = 39.9527,    # transmitter lat
    lon                = 75.1785,    # positive — judy_predictor forces negative
    class_flag         = "B",
    antenna_type       = "NON",
    aant_rotation_deg  = 0,
    uneven_polarization= 0,
)

# Tabernacle, NJ — receiver
RX_LAT = 39.8732
RX_LON = 74.6643   # positive — will be forced negative

# Calculate distance and bearing
def haversine(lat1, lon1, lat2, lon2):
    R = 3959  # miles
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def bearing(lat1, lon1, lat2, lon2):
    """Bearing from (lat1,lon1) to (lat2,lon2) in degrees."""
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(math.radians(lat2))
    y = math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) - math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

dist_mi  = haversine(WXPN['lat'], -WXPN['lon'], RX_LAT, -RX_LON)
bear_deg = bearing(RX_LAT, -RX_LON, WXPN['lat'], -WXPN['lon'])  # RX → TX

print(f"TX–RX distance : {dist_mi:.1f} miles")
print(f"Bearing (RX→TX): {bear_deg:.1f}°")

# Run JUDY
dbu = judy.predict(
    **WXPN,
    receiver_lat   = RX_LAT,
    receiver_lon   = RX_LON,
    distance_miles = dist_mi,
    bearing        = bear_deg,
    receiver_hagl  = 10.0,
)

print(f"\nJUDY prediction : {dbu:.1f} dBu")
print(f"Interpretation  : ", end="")
if   dbu >= 80: print("Excellent")
elif dbu >= 70: print("Very Good")
elif dbu >= 60: print("Good")
elif dbu >= 50: print("Fair")
elif dbu >= 40: print("Poor")
else:           print("No reliable signal")

### E1 — Sensitivity to Distance

Keep the same WXPN transmitter parameters but vary the receiver distance from 5 to 150 miles. Plot predicted field strength vs. distance.

1. At what distance does JUDY predict the signal drops below 50 dBu ("fair")? Below 40 dBu ("poor")?
2. Does the decay look linear, logarithmic, or something else? What do you know from physics about how radio signals decay with distance?
3. How does changing `haat` from 174 m to 50 m affect the distance-decay curve?

In [ ]:
distances = np.linspace(5, 150, 100)
# For a receiver due east of WXPN (bearing ≈ 90° from RX perspective)
preds_nominal = [judy.predict(**WXPN, receiver_lat=WXPN['lat'], receiver_lon=WXPN['lon'] - d/53,
                              distance_miles=d, bearing=270, receiver_hagl=10) for d in distances]
preds_low_haat = [judy.predict(**{**WXPN, 'haat': 50}, receiver_lat=WXPN['lat'],
                               receiver_lon=WXPN['lon'] - d/53,
                               distance_miles=d, bearing=270, receiver_hagl=10) for d in distances]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(distances, preds_nominal,  label=f"HAAT = {WXPN['haat']} m (actual)")
ax.plot(distances, preds_low_haat, label="HAAT = 50 m", linestyle='--')
ax.axhline(50, color='orange', linestyle=':', label='50 dBu (Fair threshold)')
ax.axhline(40, color='red',    linestyle=':', label='40 dBu (Poor threshold)')
ax.set_xlabel('Distance (miles)')
ax.set_ylabel('JUDY Predicted Field Strength (dBu)')
ax.set_title('WXPN 88.5 — JUDY Field Strength vs. Distance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find crossover distances
for thresh, label in [(50, 'Fair'), (40, 'Poor')]:
    crosses = [distances[i] for i in range(len(preds_nominal)-1)
               if preds_nominal[i] >= thresh > preds_nominal[i+1]]
    if crosses:
        print(f"Drops below {thresh} dBu ({label}) at ~{crosses[0]:.0f} miles")

---
## Part 4: Batch Predictions — JUDY's Speed Advantage

The real value of JUDY is speed. The `predict_batch()` method sends all inputs to the XGBoost model in a single vectorized call — no Python loop, no per-call overhead. Let's time it against a loop of single calls.

In [ ]:
# Build 10,000 prediction requests: WXPN at random receiver locations
rng = np.random.default_rng(42)
n = 10_000

rx_lats = rng.uniform(37, 42, n)
rx_lons = rng.uniform(72, 79, n)   # positive — predictor forces negative

batch_inputs = [
    dict(
        **WXPN,
        receiver_lat   = float(rx_lats[i]),
        receiver_lon   = float(rx_lons[i]),
        distance_miles = haversine(WXPN['lat'], -WXPN['lon'], rx_lats[i], -rx_lons[i]),
        bearing        = bearing(rx_lats[i], -rx_lons[i], WXPN['lat'], -WXPN['lon']),
        receiver_hagl  = 10.0,
    )
    for i in range(n)
]

# --- Batch (fast) ---
t0 = time.perf_counter()
batch_results = judy.predict_batch(batch_inputs)
batch_time = time.perf_counter() - t0

print(f"Batch: {n:,} predictions in {batch_time:.3f}s  ({n/batch_time:,.0f} predictions/sec)")

# --- Loop (slow) — just 500 to estimate ---
n_loop = 500
t0 = time.perf_counter()
for inp in batch_inputs[:n_loop]:
    judy.predict(**inp)
loop_time = time.perf_counter() - t0
estimated_full = loop_time / n_loop * n

print(f"Loop:  {n_loop:,} predictions in {loop_time:.3f}s  (est. {estimated_full:.1f}s for {n:,})")
print(f"Speedup: ~{estimated_full/batch_time:.0f}x")

In [ ]:
# Visualize the 10K predictions as a scatter plot colored by field strength
results_arr = np.array([r if r is not None else np.nan for r in batch_results])

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(-rx_lons, rx_lats, c=results_arr, cmap='RdYlGn',
                vmin=20, vmax=90, s=3, alpha=0.6)
ax.scatter(-WXPN['lon'], WXPN['lat'], color='blue', s=100, zorder=5, label='WXPN TX')
plt.colorbar(sc, ax=ax, label='JUDY Field Strength (dBu)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'WXPN 88.5 — JUDY Coverage Estimate ({n:,} receiver points)')
ax.legend()
plt.tight_layout()
plt.show()

### E2 — Coverage Radius Estimation

Using `batch_results` from above:

1. What fraction of the 10,000 receiver points are predicted to receive ≥ 50 dBu from WXPN?
2. What is the maximum distance at which JUDY predicts ≥ 50 dBu? Does that match what you found in E1?
3. Does the predicted coverage look symmetric around the transmitter? What real-world factors would make actual coverage asymmetric that JUDY cannot account for?

In [ ]:
# YOUR CODE HERE
# Hint: compute distances for each of the 10K points using haversine()
# Then filter by field_strength >= 50 and find the max distance

distances_all = np.array([
    haversine(WXPN['lat'], -WXPN['lon'], rx_lats[i], -rx_lons[i])
    for i in range(n)
])

covered_mask = results_arr >= 50
print(f"Fraction covered (≥50 dBu): {covered_mask.sum()/n:.1%}")
print(f"Max distance with ≥50 dBu : {distances_all[covered_mask].max():.1f} miles")

---
## Part 5: Feature Importance

XGBoost records how much each feature contributed to reducing prediction error across all the trees. Let's inspect that.

In [ ]:
from judy_predictor import _FEATURE_ORDER

# XGBoost Booster stores feature importances
booster = judy.model  # the loaded sklearn wrapper or raw booster

# Handle both sklearn Pipeline wrapper and raw XGBoost model
try:
    importances = booster.feature_importances_
except AttributeError:
    importances = np.array(list(booster.get_booster().get_fscore().values()))

if len(importances) == len(_FEATURE_ORDER):
    feat_df = pd.DataFrame({'feature': _FEATURE_ORDER, 'importance': importances})
    feat_df = feat_df.sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feat_df['feature'], feat_df['importance'])
    ax.set_xlabel('Feature Importance (gain)')
    ax.set_title('JUDY — XGBoost Feature Importances')
    plt.tight_layout()
    plt.show()
else:
    print(f"Feature count mismatch ({len(importances)} vs {len(_FEATURE_ORDER)}); skipping plot.")
    print("Model may use a different internal feature order — try: judy.model.get_booster().get_score()")

### E3 — Interpreting Feature Importance

1. Which features does JUDY rely on most? Which does it rely on least? Does that match your physical intuition?
2. `bearing` is one of the features. Why might the direction from receiver to transmitter matter for a nominally omnidirectional station?
3. JUDY has no terrain feature — no elevation profile, no obstacle data. Based on the feature importance chart, how does JUDY approximate terrain effects indirectly?

---
## Part 6: Applying JUDY to NWR — Frequency Extrapolation

JUDY was trained on FM stations at 88.1–108.1 MHz. NWR transmits at 162.4–162.55 MHz — about 55 MHz higher. When you pass a NWR frequency into `judy_predictor.py`, the min-max normalization clips it to the training maximum.

Concretely: for any frequency ≥ 108.1 MHz, the normalized value becomes `1.0` — the same as a 108.1 MHz station. All NWR frequencies are treated identically by the model.

This is a known limitation. However, the other 15 features (power, height, distance, etc.) are still meaningful. The question for research is: **how wrong is JUDY for NWR, and in what direction?**

Higher frequencies generally attenuate faster with distance — so JUDY, treating NWR as 108 MHz, might *overestimate* real NWR field strength. Let's test this by comparing JUDY against the RadioLand API for a set of NWR stations.

In [ ]:
WX_URL = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/wx_stations.csv"
wx = pd.read_csv(WX_URL)
wx = wx[wx['country'] == 'USA'].copy()
wx['longitude'] = -wx['longitude'].abs()

print(f"{len(wx)} NWR stations loaded.")
wx[['callsign', 'city', 'state', 'frequency', 'erp', 'haat', 'latitude', 'longitude']].head()

In [ ]:
# RadioLand API function (from Week 3)
RADIOLAND_BASE = "http://52.151.197.43/search_stream"

def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=False):
    params = dict(lat=lat, lon=lon, search_freq="none", callsign="none",
                  request_type=1, pi_code="none", sig_strength=min_sig_strength,
                  am_sig_strength=2, startMiles="none", miles="null",
                  slogan="none", owner="none", format="none", wfo="none",
                  rxHeight=rx_height, mlbTeam="none", market="none",
                  country="none", sp="none", measurementUnit="metric",
                  locationName="", broadcastBand="WX", timeOfDay="day", model="longley_rice")
    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=90)
        resp.raise_for_status()
    except requests.RequestException as e:
        return None
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue
        event = json.loads(line[6:])
        if event.get('type') == 'complete':
            result = json.loads(event['data'])
            df = pd.DataFrame(result['data'])
            if not df.empty:
                for col in ['lon', 'transmitter_lon']:
                    if col in df.columns:
                        df[col] = -df[col].abs()
            return df
    return None

In [ ]:
# Pick a handful of receiver cities and compare JUDY vs API for the best NWR station at each
# Using a small set to keep runtime reasonable

test_locations = [
    {"name": "Tabernacle, NJ",  "lat": 39.8732, "lon": -74.6643},
    {"name": "Chicago, IL",      "lat": 41.8781, "lon": -87.6298},
    {"name": "Phoenix, AZ",      "lat": 33.4484, "lon": -112.0740},
    {"name": "Bozeman, MT",      "lat": 45.6770, "lon": -111.0429},
    {"name": "New Orleans, LA",  "lat": 29.9511, "lon": -90.0715},
]

comparison_rows = []

for loc in test_locations:
    print(f"Querying {loc['name']}...", end=' ', flush=True)
    api_df = query_nwr_coverage(loc['lat'], loc['lon'])
    if api_df is None or api_df.empty:
        print("no API results")
        continue

    # Take the top 3 stations by API field strength and run JUDY on each
    for _, row in api_df.head(3).iterrows():
        # Find transmitter coords — API returns transmitter_lat/lon when available
        tx_lat = row.get('transmitter_lat', row.get('lat', None))
        tx_lon = row.get('transmitter_lon', row.get('lon', None))
        if tx_lat is None or tx_lon is None:
            continue

        dist_mi  = haversine(tx_lat, tx_lon, loc['lat'], loc['lon'])
        bear_deg = bearing(loc['lat'], loc['lon'], tx_lat, tx_lon)

        judy_dbu = judy.predict(
            frequency          = float(row.get('frequency', 162.4)),
            erp                = float(row.get('erp', 1000)),
            haat               = float(row.get('haat', 100)),
            hagl               = float(row.get('hagl', 50)),
            amsl               = float(row.get('amsl', 0)),
            lat                = tx_lat,
            lon                = abs(tx_lon),  # predictor forces negative
            class_flag         = str(row.get('class_flag', 'C')),
            antenna_type       = str(row.get('antenna_type', 'NON')),
            aant_rotation_deg  = float(row.get('aant_rotation_deg', 0)),
            uneven_polarization= float(row.get('uneven_polarization', 0)),
            receiver_lat       = loc['lat'],
            receiver_lon       = abs(loc['lon']),
            distance_miles     = dist_mi,
            bearing            = bear_deg,
            receiver_hagl      = 10.0,
        )

        comparison_rows.append({
            'location':    loc['name'],
            'callsign':    row.get('callsign', ''),
            'frequency':   row.get('frequency', ''),
            'distance_mi': round(dist_mi, 1),
            'api_dbu':     round(float(row['field_strength']), 1),
            'judy_dbu':    round(judy_dbu, 1) if judy_dbu is not None else None,
        })

    print("done")
    time.sleep(2)

comp_df = pd.DataFrame(comparison_rows)
comp_df['error_dbu'] = comp_df['judy_dbu'] - comp_df['api_dbu']
print(f"\n{len(comp_df)} station–location pairs compared.")
comp_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: JUDY vs API
ax = axes[0]
ax.scatter(comp_df['api_dbu'], comp_df['judy_dbu'])
lim = [comp_df[['api_dbu','judy_dbu']].min().min() - 5,
       comp_df[['api_dbu','judy_dbu']].max().max() + 5]
ax.plot(lim, lim, 'k--', label='Perfect agreement')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('RadioLand API (dBu)')
ax.set_ylabel('JUDY (dBu)')
ax.set_title('JUDY vs. RadioLand API — NWR Stations')
ax.legend()

# Error distribution
ax = axes[1]
ax.hist(comp_df['error_dbu'].dropna(), bins=10, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', label='Zero error')
ax.axvline(comp_df['error_dbu'].mean(), color='orange', linestyle='--',
           label=f"Mean error: {comp_df['error_dbu'].mean():.1f} dBu")
ax.set_xlabel('JUDY − API error (dBu)')
ax.set_ylabel('Count')
ax.set_title('JUDY Prediction Error Distribution')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Mean error (JUDY − API): {comp_df['error_dbu'].mean():.1f} dBu")
print(f"MAE                    : {comp_df['error_dbu'].abs().mean():.1f} dBu")
print(f"JUDY overestimates     : {(comp_df['error_dbu'] > 0).mean():.0%} of cases")

### E4 — NWR Extrapolation Analysis

1. Does JUDY systematically over- or under-estimate NWR field strength compared to the RadioLand API? By how much on average?
2. The frequency clipping means all 7 NWR frequencies are treated identically by JUDY. How could you test whether this actually causes error — or whether the other features compensate?
3. Given the error you measured, could you apply a simple **bias correction** (e.g., subtract the mean error) to make JUDY more accurate for NWR? What are the limitations of that approach?
4. For the gap analysis in Weeks 4–5, would you trust JUDY enough to use it as a fast pre-screening tool before calling the API? Justify your threshold.

In [ ]:
# YOUR CODE HERE
# E4.2 hint: try grouping by frequency and computing mean error per frequency
print(comp_df.groupby('frequency')['error_dbu'].mean())

# E4.3 hint: apply bias correction and recompute MAE
mean_bias = comp_df['error_dbu'].mean()
corrected = comp_df['judy_dbu'] - mean_bias
corrected_mae = (corrected - comp_df['api_dbu']).abs().mean()
print(f"\nMAE after bias correction: {corrected_mae:.1f} dBu  (was {comp_df['error_dbu'].abs().mean():.1f} dBu)")

---
## Part 7: Research Application — Fast Grid Sweep

Here's the payoff. Using the RadioLand API to evaluate NWR coverage for every 5×5 km grid cell in the US would take weeks. Using JUDY, you can sweep the entire country in minutes — with the caveat you characterized above.

The cell below runs a coarse grid sweep over the contiguous US and flags cells that JUDY predicts are likely gaps. This is a **screening tool**, not a final answer — you'd follow up flagged areas with API calls to confirm.

In [ ]:
# Coarse grid: 0.5° spacing (~35 miles) over CONUS
GRID_STEP = 0.5
grid_lats = np.arange(25, 50, GRID_STEP)
grid_lons = np.arange(-125, -66, GRID_STEP)

lat_grid, lon_grid = np.meshgrid(grid_lats, grid_lons)
lat_flat = lat_grid.ravel()
lon_flat = lon_grid.ravel()
print(f"Grid points: {len(lat_flat):,}")

# For each grid point, find the closest NWR station
from scipy.spatial import cKDTree

wx_lats = wx['latitude'].values
wx_lons = wx['longitude'].values

# Build KD-tree on (lat, lon) — rough distance proxy (ignores earth curvature, fine for nearest-neighbor)
tree = cKDTree(np.column_stack([wx_lats, wx_lons]))
dists_deg, idxs = tree.query(np.column_stack([lat_flat, lon_flat]), k=1)

# Build batch inputs for JUDY: each grid point vs. its nearest NWR station
batch = []
for i in range(len(lat_flat)):
    tx = wx.iloc[idxs[i]]
    dist_mi  = haversine(float(tx['latitude']), float(tx['longitude']), lat_flat[i], lon_flat[i])
    bear_deg = bearing(lat_flat[i], lon_flat[i], float(tx['latitude']), float(tx['longitude']))
    batch.append(dict(
        frequency           = float(tx.get('frequency', 162.4)),
        erp                 = float(tx.get('erp', 1000)),
        haat                = float(tx.get('haat', 100)),
        hagl                = float(tx.get('hagl', 50)),
        amsl                = 0.0,
        lat                 = float(tx['latitude']),
        lon                 = abs(float(tx['longitude'])),
        class_flag          = 'C',
        antenna_type        = 'NON',
        aant_rotation_deg   = 0.0,
        uneven_polarization = 0.0,
        receiver_lat        = lat_flat[i],
        receiver_lon        = abs(lon_flat[i]),
        distance_miles      = dist_mi,
        bearing             = bear_deg,
        receiver_hagl       = 10.0,
    ))

t0 = time.perf_counter()
sweep_results = judy.predict_batch(batch)
elapsed = time.perf_counter() - t0
print(f"Swept {len(batch):,} grid points in {elapsed:.2f}s")

In [ ]:
sweep_arr = np.array([r if r is not None else np.nan for r in sweep_results])

fig, ax = plt.subplots(figsize=(14, 8))
sc = ax.scatter(lon_flat, lat_flat, c=sweep_arr, cmap='RdYlGn',
                vmin=20, vmax=80, s=4, alpha=0.7)
plt.colorbar(sc, ax=ax, label='JUDY Best NWR Field Strength (dBu)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('JUDY Coarse NWR Coverage Sweep — CONUS (nearest station per grid point)')
ax.set_facecolor('#e0e8f0')

# Highlight potential gaps
gap_mask = sweep_arr < 50
ax.scatter(lon_flat[gap_mask], lat_flat[gap_mask],
           color='black', s=8, alpha=0.3, label=f'Potential gaps < 50 dBu ({gap_mask.sum():,} cells)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"Potential gap cells (< 50 dBu): {gap_mask.sum():,} of {len(sweep_arr):,} ({gap_mask.mean():.1%})")

### E5 — Reading the Coverage Map

1. Which regions show the most potential gaps? Do these match your intuition from the NWR station density map you built in earlier weeks?
2. This sweep only considers the **nearest** NWR station. In reality, a receiver might pick up a more powerful station that is farther away. How would you modify the sweep to find the **best** (highest predicted field strength) station within some radius — not just the nearest?
3. Given the JUDY–API error you measured in Part 6, a cell predicted at 47 dBu might actually be covered (if JUDY underestimates by ~5 dBu) or might be a genuine gap. How would you design a follow-up sampling strategy to confirm the true gap cells using the RadioLand API?

In [ ]:
# YOUR CODE HERE
# Suggested approach for E5.2: for each grid point, query the K nearest stations
# and take the maximum JUDY prediction across all K.

# Example: top 5 nearest stations per grid point
K = 5
_, idxs_k = tree.query(np.column_stack([lat_flat, lon_flat]), k=K)

# Build batch for all K candidates
# ... YOUR CODE ...

---
## Reflection

Write brief answers below.

1. What are the three most important limitations of using JUDY for NWR gap analysis?
2. JUDY was trained on FM measurements. What would it take to train a proper NWR version of the model? What training data would you need, and where could you get it?
3. You now have two tools: RadioLand API (slow, accurate, NWR-native) and JUDY (fast, approximate, FM-trained). Describe a two-stage workflow that uses both intelligently for the national gap analysis.
4. Feature importance showed which inputs JUDY uses most. If you were retraining JUDY from scratch for NWR, what additional features would you want to include that are currently absent?

**Your answers:**

1. 
2. 
3. 
4. 